In [ ]:
!apt-get -qq update && apt-get -qq install -y tesseract-ocr tesseract-ocr-spa
%pip install -q nltk==3.9.2 pytesseract pillow scikit-learn matplotlib seaborn


In [ ]:
from google.colab import files
from pathlib import Path
from zipfile import ZipFile
import os, shutil

subidos_documentos = files.upload()  # Selecciona documentos.zip
archivos_zip = [nombre for nombre in subidos_documentos if nombre.lower().endswith('.zip')]
if len(archivos_zip) != 1:
    raise ValueError("Sube un solo archivo documentos.zip.")
ruta_exterior = Path('/content/documentos.zip')
ruta_exterior.write_bytes(subidos_documentos[archivos_zip[0]])
DOCUMENTOS_DIR = Path('/content/documentos')
if DOCUMENTOS_DIR.exists():
    shutil.rmtree(DOCUMENTOS_DIR)
DOCUMENTOS_DIR.mkdir(parents=True, exist_ok=True)
with ZipFile(ruta_exterior) as archivo_zip:
    raiz = DOCUMENTOS_DIR.resolve()
    for miembro in archivo_zip.infolist():
        destino = (raiz / miembro.filename).resolve()
        if destino != raiz and raiz not in destino.parents:
            raise ValueError(f"El ZIP contiene una ruta no permitida: {miembro.filename}")
    archivo_zip.extractall(raiz)
os.environ['BOT_DOCUMENTS_DIR'] = str(DOCUMENTOS_DIR)
requeridos = ['dataset_ttn.zip', 'bitacora.zip', 'frases_intenciones.csv', 'Calculo_KPIs.csv']
for nombre in requeridos:
    encontrados = [p for p in DOCUMENTOS_DIR.rglob('*') if p.is_file() and p.name.lower() == nombre.lower()]
    if len(encontrados) != 1:
        raise FileNotFoundError(f"Se esperaba una sola copia de {nombre} dentro de documentos.zip; encontré {len(encontrados)}.")
print('Paquete cargado. Archivos localizados:')
for nombre in requeridos:
    print('-', nombre)


In [ ]:
%%writefile bot_kpis.py
"""Bot de KPIs NOC basado en reglas. Python 3.10+ y NLTK.

Las respuestas normales muestran solo periodo y resultados por severidad.
"""
from __future__ import annotations

import argparse
import csv
import io
import re
import shutil
import unicodedata
from collections import Counter
from dataclasses import dataclass
from datetime import datetime, timedelta, timezone
from pathlib import Path

try:
    from nltk.tokenize import wordpunct_tokenize
except ImportError:
    wordpunct_tokenize = None

ARCHIVO = "Calculo_KPIs.csv"
COLUMNAS = "CODIGO_FOLIO DESC_FOLIO DESC_ORIGEN_FOLIO_NORM DESC_PRIORIDAD TIPO DESC_GRUPO_TRABAJO REGION FECHA_APERTURA FECHA_INICIO_FALLA TIEMPO_VIDA TIEMPO_APERTURA MINUTOS_AP SEMANA STATUS_REAL STATUS_REV OBSERVACIONES ANO MES DIAT TBCKL DESC_ESTADO DESC_SUBESTADO FECHA_INICIO_TTC FECHA_HORA_CIERRE HORA_CIERRE ORIGEN_FECHA_CIERRE ORIGEN_INICIO_TTC TTC_MINUTOS TTC VALIDACION_TTC TTN RED_SEV TTR SOLICITA_NOMBRE_CIERRE MOD_CIERRE RN".split()
ORIGENES = {"rose", "gestion de eventos", "ccr mexico transmision", "ccr mexico plataformas", "ccr mexico datos", "ccr mexico acceso"}
MESES = dict(zip("enero febrero marzo abril mayo junio julio agosto septiembre octubre noviembre diciembre".split(), range(1, 13)))
MESES["setiembre"] = 9
SIN_FOLIOS = "No se cuentan con folios de esta severidad para realizar el cálculo"
FORMATOS = ("%d/%m/%Y %H:%M:%S", "%d/%m/%Y %H:%M", "%d/%m/%y %H:%M:%S", "%Y-%m-%d %H:%M:%S", "%Y-%m-%dT%H:%M:%S", "%d/%m/%Y", "%d/%m/%y", "%Y-%m-%d")


def normalizar(texto):
    """Quita diferencias de acentos, espacios y mayúsculas para comparar textos."""
    limpio = "".join(c for c in unicodedata.normalize("NFKD", str(texto)) if not unicodedata.combining(c))
    return " ".join(limpio.casefold().split())


def ahora_mexico():
    """Obtiene la hora de Ciudad de México, sin depender de la zona del equipo."""
    return datetime.now(timezone(timedelta(hours=-6))).replace(tzinfo=None)


def fecha(valor):
    """Interpreta formatos de fecha conocidos; devuelve None si el dato es inválido."""
    for formato in FORMATOS:
        try:
            return datetime.strptime(str(valor or "").strip(), formato)
        except ValueError:
            pass
    return None


def resolver_ruta_csv(ruta=None):
    """Acepta una ruta de archivo o busca Calculo_KPIs.csv dentro de una carpeta."""
    valor = str(ruta).strip().strip("\"'") if ruta is not None else str(Path(__file__).resolve().parent / ARCHIVO)
    destino = Path(valor).expanduser()
    return destino / ARCHIVO if destino.is_dir() else destino


def valida_archivo(ruta=None):
    """Comprueba si el CSV existe junto al programa o en la ruta indicada."""
    return resolver_ruta_csv(ruta).is_file()


def leer_csv(ruta):
    """Lee el CSV y comprueba encabezados, número de campos y folios únicos."""
    ruta = resolver_ruta_csv(ruta)
    if not ruta.is_file():
        raise ValueError(f"No se encontró el archivo CSV: {ruta}. Indica el archivo completo o la carpeta que contiene {ARCHIVO}.")
    if ruta.suffix.lower() != ".csv":
        raise ValueError("Se requiere un archivo .csv; no selecciones un libro .xlsx.")
    datos = ruta.read_bytes()
    try:
        texto = datos.decode("utf-8-sig")
    except UnicodeDecodeError:
        texto = datos.decode("cp1252")
    try:
        separador = csv.Sniffer().sniff(texto[:65536], delimiters=",;\t").delimiter
    except csv.Error:
        separador = ","
    lector = csv.DictReader(io.StringIO(texto, newline=""), delimiter=separador)
    encabezados = [c.strip().upper() for c in lector.fieldnames or []]
    if len(encabezados) != len(set(encabezados)):
        raise ValueError("El CSV contiene columnas duplicadas.")
    if set(encabezados) != set(COLUMNAS):
        raise ValueError(f"Estructura incorrecta. Faltan: {sorted(set(COLUMNAS)-set(encabezados))}. Sobran: {sorted(set(encabezados)-set(COLUMNAS))}.")
    filas, vistos = [], set()
    for numero, r in enumerate(lector, 2):
        if None in r or any(v is None for v in r.values()):
            raise ValueError(f"La fila lógica {numero} no tiene 36 campos.")
        r = {k.strip().upper(): v.strip() for k, v in r.items()}
        codigo = r["CODIGO_FOLIO"]
        if not re.fullmatch(r"\d{6}-\d{3,}", codigo) or fecha("20" + codigo[:2] + "-" + codigo[2:4] + "-" + codigo[4:6]) is None:
            raise ValueError(f"Folio inválido en la fila {numero}: {codigo!r}.")
        if codigo in vistos:
            raise ValueError(f"Folio duplicado: {codigo}. Consolida la base antes de calcular.")
        vistos.add(codigo)
        filas.append(r)
    if not filas:
        raise ValueError("El CSV no contiene registros.")
    return filas


def valida_estructura(ruta):
    """Indica si el archivo tiene la estructura requerida y explica los errores."""
    try:
        filas = leer_csv(ruta)
        return True, f"Archivo válido: {len(filas):,} registros y 36 columnas."
    except PermissionError:
        return False, "No se pudo leer el archivo por permisos. Comprueba que sea un CSV accesible y que tengas permiso de lectura."
    except (ValueError, OSError, csv.Error, UnicodeError) as e:
        return False, str(e)


def carga_archivo(destino=None, colab=False, entrada=input):
    """Solicita un CSV y lo activa únicamente después de validarlo."""
    destino = resolver_ruta_csv(destino)
    destino.parent.mkdir(parents=True, exist_ok=True)
    while True:
        if colab:
            from google.colab import files
            print("Selecciona un solo CSV o cancela la selección.")
            cargados = files.upload(target_dir=str(destino.parent / "cargas"))
            if not cargados:
                return None
            if len(cargados) != 1:
                print("Selecciona solo un archivo CSV.")
                continue
            nombre, datos = next(iter(cargados.items()))
            if Path(nombre).suffix.lower() != ".csv":
                print("Se requiere un CSV, no un archivo XLSX.")
                continue
            origen = destino.with_suffix(".validacion.csv")
            origen.write_bytes(datos)
        else:
            respuesta = entrada("Ruta completa del CSV o carpeta que lo contiene (o cancelar): ").strip().strip("\"'")
            if normalizar(respuesta) in {"", "cancelar", "salir"}:
                return None
            origen = resolver_ruta_csv(respuesta)
        correcto, mensaje = valida_estructura(origen)
        print(mensaje)
        if not correcto:
            continue
        if origen.resolve() != destino.resolve():
            try:
                shutil.copyfile(origen, destino)
            except OSError as e:
                print(f"El CSV es válido, pero no se pudo copiar a {destino}: {e}. Revisa los permisos de esa carpeta o cancela y usa --archivo con la ruta del CSV.")
                continue
        if colab and origen != destino:
            origen.unlink(missing_ok=True)
        return destino


def Intrucciones_bot():
    """Presenta una explicación breve del bot y ejemplos de consultas."""
    return ("BOT DE KPIs NOC — TTT, TTC y TTN\n"
            "Consulta por hora, día, semana o mes el estatus de los KPIs principales.\n"
            "Ejemplos: ¿Cuál es el KPI TTT del 15 de octubre de 2026?\n"
            "¿Cuál es el estatus del folio 261013-001?\n"
            "Puedes filtrar por tipo manual/automático y severidad SF1–SF4.\n"
            "Otros comandos: ayuda, datos, detalle, exportar y cargar archivo.\n"
            "Para terminar: adiós, bye, hasta luego, cya, ciao o salir.")


def saludo_inicial():
    """Da la bienvenida e invita a realizar una consulta."""
    return "¡Hola! Soy tu asistente de KPIs del NOC. ¿Qué deseas consultar?"


def despedida_bot():
    """Devuelve el mensaje de cierre de la conversación."""
    return "Hasta luego. Gracias por consultar los KPIs del NOC."


@dataclass
class Medicion:
    """Guarda el resultado de evaluar un folio, sin modificar sus datos originales."""
    fila: dict
    estado: str
    minutos: float | None = None
    motivo: str = ""
    discrepancia: bool = False


def severidad(fila):
    """Extrae SF1, SF2, SF3 o SF4 de la prioridad del folio."""
    m = re.search(r"\bSF[1-4]\b", fila.get("DESC_PRIORIDAD", ""), re.I)
    return m.group().upper() if m else "Sin severidad"


def calcular_ttt(fila):
    """Evalúa el tiempo entre la alarma y la creación del ticket según severidad."""
    sev = severidad(fila)
    if sev == "Sin severidad":
        return Medicion(fila, "Sin datos", motivo="Severidad no reconocida")
    inicio, fin = fecha(fila.get("FECHA_INICIO_FALLA")), fecha(fila.get("FECHA_APERTURA"))
    if inicio is None or fin is None:
        return Medicion(fila, "Sin datos", motivo="Fecha de alarma o apertura ausente/inválida")
    # Apertura menos alarma: la resta impresa en el Word está invertida.
    minutos = (fin - inicio).total_seconds() / 60
    if minutos < 0:
        return Medicion(fila, "Fecha inconsistente", minutos, "La apertura es anterior a la alarma")
    limite = 10 if sev in {"SF1", "SF2"} else 30
    return Medicion(fila, "En tiempo" if minutos <= limite else "Fuera de tiempo", minutos)


def calcular_ttc(fila):
    """Evalúa el cierre en hasta 20 minutos, solo para los estados permitidos."""
    if normalizar(fila.get("DESC_ESTADO", "")) not in {"folio cerrado", "solicitud cierre"}:
        return Medicion(fila, "No aplica", motivo="Estado no elegible para TTC")
    referencia = normalizar(fila.get("TTC", ""))
    if referencia == "no aplica":
        return Medicion(fila, "No aplica", motivo="TTC indica No aplica")
    inicio, fin = fecha(fila.get("FECHA_INICIO_TTC")), fecha(fila.get("FECHA_HORA_CIERRE"))
    if inicio is None or fin is None:
        return Medicion(fila, "Sin datos", motivo="Inicio TTC o cierre ausente/inválido")
    minutos = (fin - inicio).total_seconds() / 60
    if minutos < 0:
        return Medicion(fila, "Fecha inconsistente", minutos, "El cierre es anterior al inicio TTC")
    cumple = minutos <= 20
    esperado = "dentro sla" if cumple else "fuera de sla"
    return Medicion(fila, "En tiempo" if cumple else "Fuera de tiempo", minutos,
                    discrepancia=referencia != esperado)


def calcular_ttn(fila):
    """Usa la columna TTN: solo Dentro de SLA cuenta como cumplimiento."""
    cumple = normalizar(fila.get("TTN", "")) == "dentro de sla"
    return Medicion(fila, "En tiempo" if cumple else "Fuera de tiempo", motivo="Etiqueta de TTN; fase 1")


CALCULOS = {"TTT": calcular_ttt, "TTC": calcular_ttc, "TTN": calcular_ttn}


def resumen_mediciones(mediciones):
    """Cuenta resultados y calcula el porcentaje sin redondear antes de evaluar la meta."""
    estados = Counter(m.estado for m in mediciones)
    dentro, fuera = estados["En tiempo"], estados["Fuera de tiempo"]
    total = dentro + fuera
    return {"dentro": dentro, "fuera": fuera, "evaluables": total,
            "sin_datos": estados["Sin datos"], "inconsistentes": estados["Fecha inconsistente"],
            "no_aplica": estados["No aplica"], "porcentaje": 100 * dentro / total if total else None,
            "cumple": dentro * 100 >= total * 95 if total else None,
            "discrepancias": sum(m.discrepancia for m in mediciones)}


def grupo_trabajo(mediciones):
    """Ordena los grupos por folios fuera de tiempo cuando se pide ese detalle."""
    grupos = {m.fila["DESC_GRUPO_TRABAJO"] or "Sin grupo" for m in mediciones}
    resumen = [(g, resumen_mediciones([m for m in mediciones if (m.fila["DESC_GRUPO_TRABAJO"] or "Sin grupo") == g])) for g in grupos]
    return sorted(resumen, key=lambda item: (-item[1]["fuera"], item[0]))


def estatus_folio(filas, codigo):
    """Devuelve únicamente DESC_ESTADO, como solicita el Word actualizado."""
    if not re.fullmatch(r"\d{6}-\d{3,}", codigo):
        return "Indica un folio válido; por ejemplo, 261013-001."
    encontrados = [r for r in filas if r["CODIGO_FOLIO"] == codigo]
    if not encontrados:
        return "No encontré ese folio. Verifica el número y reformula la consulta o carga una base actualizada."
    return encontrados[0]["DESC_ESTADO"] or "No hay estado disponible para ese folio."


def identificar_intencion(texto):
    """Reconoce la consulta y extrae KPI, tipo de folio, severidad o número de ticket."""
    if wordpunct_tokenize is None:
        raise RuntimeError("Instala NLTK: python -m pip install -r requirements.txt")
    t = normalizar(texto)
    tokens = set(wordpunct_tokenize(t))
    if re.fullmatch(r"(?:adios|bye|hasta luego|cya|ciao|salir|terminar)[.!\s]*", t):
        return {"accion": "despedida"}
    for opciones, accion in [({"ayuda", "instrucciones", "menu", "ejemplos"}, "ayuda"),
                             ({"hola", "buenos dias", "buenas tardes"}, "saludo"),
                             ({"datos", "cobertura", "diagnostico"}, "datos")]:
        if t in opciones:
            return {"accion": accion}
    if tokens & {"cargar", "recargar"}:
        return {"accion": "cargar"}
    if "exportar" in tokens:
        return {"accion": "exportar"}
    folio = re.search(r"\b\d{6}-\d{3,}\b", t)
    if folio or tokens & {"estatus", "estado"} and tokens & {"folio", "ticket", "fg"}:
        return {"accion": "estatus", "folio": folio.group() if folio else None}
    kpis = [k for k in CALCULOS if k.lower() in tokens]
    for expresion, kpi in {"time to ticket": "TTT", "time to close": "TTC", "time to notify": "TTN"}.items():
        if expresion in t and kpi not in kpis:
            kpis.append(kpi)
    if "los tres" in t or tokens & {"todos"} and tokens & {"kpis", "indicadores"}:
        kpis = list(CALCULOS)
    automatico = bool(tokens & {"automatico", "automaticos", "automatica", "automaticas"})
    manual = bool(tokens & {"manual", "manuales"})
    tipo = "Automatico" if automatico and not manual else "Manual" if manual and not automatico else None
    sev = re.search(r"\bsf\s*([1-4])\b", t)
    relevante = bool(kpis or tokens & {"kpi", "kpis", "sla", "ans", "cumplimiento", "detalle", "detalles", "grupo", "grupos"})
    accion = "grupos" if tokens & {"grupo", "grupos"} else "detalle" if tokens & {"detalle", "detalles", "listado"} else "kpi"
    return {"accion": accion if relevante else "desconocida",
            "kpis": kpis, "tipo": tipo, "severidad": "SF" + sev.group(1) if sev else None}


@dataclass
class Periodo:
    """Representa un intervalo: inicio incluido y fin excluido."""
    inicio: datetime
    fin: datetime
    descripcion: str


def periodo_consulta(texto, ahora=None):
    """Convierte las fechas de la pregunta en un intervalo de consulta."""
    ahora = ahora or ahora_mexico()
    t = normalizar(texto)
    dia = ahora.replace(hour=0, minute=0, second=0, microsecond=0)
    iso = re.search(r"\b(\d{4})-(\d{2})-(\d{2})\b", t)
    barra = re.search(r"\b(\d{1,2})/(\d{1,2})/(\d{2,4})\b", t)
    nombre = re.search(r"\b(\d{1,2})\s+de\s+(" + "|".join(MESES) + r")(?:\s+(?:de\s+)?(20\d{2}))?\b", t)
    try:
        if iso:
            inicio = datetime(*map(int, iso.groups()))
        elif barra:
            d, m, a = map(int, barra.groups())
            inicio = datetime(a + 2000 if a < 100 else a, m, d)
        elif nombre:
            d, m, a = nombre.groups()
            inicio = datetime(int(a or ahora.year), MESES[m], int(d))
        else:
            inicio = None
        if inicio:
            fin, etiqueta = inicio + timedelta(days=1), "Día solicitado"
        elif "ayer" in t:
            inicio, fin, etiqueta = dia - timedelta(days=1), dia, "Ayer"
        elif "semana pasada" in t:
            fin = dia - timedelta(days=dia.weekday())
            inicio, etiqueta = fin - timedelta(days=7), "Semana pasada"
        elif "semana" in t:
            inicio, fin, etiqueta = dia - timedelta(days=dia.weekday()), ahora + timedelta(microseconds=1), "Semana actual hasta ahora"
        elif "mes pasado" in t:
            fin = dia.replace(day=1)
            inicio, etiqueta = (fin - timedelta(days=1)).replace(day=1), "Mes pasado"
        elif re.search(r"\bhoy\b|\bdia\b", t):
            inicio, fin, etiqueta = dia, ahora + timedelta(microseconds=1), "Día actual hasta ahora"
        else:
            mes = re.search(r"\b(" + "|".join(MESES) + r")(?:\s+(?:de\s+)?(20\d{2}))?\b", t)
            if mes:
                m, a = MESES[mes.group(1)], int(mes.group(2) or ahora.year)
                inicio, fin = datetime(a, m, 1), datetime(a + (m == 12), 1 if m == 12 else m + 1, 1)
                etiqueta = "Mes solicitado"
            else:
                inicio, fin, etiqueta = dia.replace(day=1), ahora + timedelta(microseconds=1), "Mes actual hasta ahora"
        if "semana" in t and (iso or barra or nombre):
            inicio -= timedelta(days=inicio.weekday())
            fin, etiqueta = inicio + timedelta(days=7), "Semana de la fecha indicada"
        hora = re.search(r"(?:de\s+(?:las\s+)?|entre\s+)(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\s*(?:a|y|hasta|al)\s*(?:las\s+)?(\d{1,2})(?::(\d{2}))?\s*(am|pm)?\b", t)
        if hora:
            h1, n1, p1, h2, n2, p2 = hora.groups()
            h1, n1 = convertir_hora(h1, n1, p1 or p2)
            h2, n2 = convertir_hora(h2, n2, p2 or p1)
            base = inicio.replace(hour=0, minute=0, second=0, microsecond=0) if (iso or barra or nombre or "ayer" in t) else dia
            inicio, fin = base.replace(hour=h1, minute=n1), base.replace(hour=h2, minute=n2)
            if fin <= inicio:
                raise ValueError("usa un intervalo ascendente dentro del mismo día")
            etiqueta = "Horario solicitado"
        elif "ultima hora" in t:
            inicio, fin, etiqueta = ahora - timedelta(hours=1), ahora + timedelta(microseconds=1), "Última hora"
        elif re.search(r"\b(hora|horas|am|pm)\b|\b\d{1,2}:\d{2}\b", t):
            raise ValueError("indica un intervalo, por ejemplo de las 10 a 11 am del 18 de octubre")
        if re.search(r"\b\d{1,2}/\d{1,2}\b", t) and not barra:
            raise ValueError("incluye el año en formato DD/MM/AAAA")
        if "entre el" in t or re.search(r"\bdel\s+\d+\s+al\s+\d+", t):
            raise ValueError("consulta un día, semana o mes a la vez")
        if re.search(r"\b(dia|del)\s+\d{1,2}\b", t) and not (nombre or barra or iso):
            raise ValueError("indica la fecha completa, por ejemplo 15 de octubre de 2026")
        return Periodo(inicio, fin, etiqueta)
    except ValueError as e:
        raise ValueError(f"No pude interpretar el periodo: {e}. Reformula la consulta.") from e


def convertir_hora(hora, minuto=None, am_pm=None):
    """Valida una hora y la convierte al formato de 24 horas."""
    h, m = int(hora), int(minuto or 0)
    if am_pm:
        if not 1 <= h <= 12:
            raise ValueError("la hora am/pm debe estar entre 1 y 12")
        h = h % 12 + (12 if am_pm == "pm" else 0)
    if not 0 <= h <= 23 or not 0 <= m <= 59:
        raise ValueError("hora inválida")
    return h, m


def describir_periodo(periodo):
    """Escribe el periodo en un formato corto y fácil de leer."""
    fin = periodo.fin
    if fin.microsecond:
        fin -= timedelta(microseconds=1)
    elif fin.time() == datetime.min.time() and periodo.inicio.time() == datetime.min.time():
        # Un día completo se presenta como fecha, sin confundir el límite exclusivo.
        ultimo = fin - timedelta(days=1)
        if ultimo.date() == periodo.inicio.date():
            return f"Periodo: {periodo.inicio:%d/%m/%Y} (día completo)"
        return f"Periodo: del {periodo.inicio:%d/%m/%Y} al {ultimo:%d/%m/%Y} (días completos)"
    return f"Periodo: del {periodo.inicio:%d/%m/%Y %H:%M} al {fin:%d/%m/%Y %H:%M}"


class BotKPIs:
    """Organiza los datos, interpreta consultas y prepara respuestas breves."""

    def __init__(self, ruta=None, reloj=None, campo_periodo="FECHA_APERTURA"):
        """Carga la base y conserva únicamente los orígenes NOC para los cálculos."""
        self.ruta = resolver_ruta_csv(ruta)
        self.filas = leer_csv(self.ruta)
        self.noc = [r for r in self.filas if normalizar(r["DESC_ORIGEN_FOLIO_NORM"]) in ORIGENES]
        self.reloj = reloj or ahora_mexico
        if campo_periodo not in {"FECHA_APERTURA", "FECHA_HORA_CIERRE"}:
            raise ValueError("Campo de periodo no admitido")
        self.campo_periodo = campo_periodo
        self.pendiente = None
        self.ultima = None
        self.historial = []
        self.terminado = False

    def datos(self):
        """Muestra la cobertura de la base cuando el usuario pide información de los datos."""
        fechas = [f for r in self.noc if (f := fecha(r["FECHA_APERTURA"]))]
        rango = f"{min(fechas):%d/%m/%Y %H:%M} a {max(fechas):%d/%m/%Y %H:%M}" if fechas else "Sin fechas válidas"
        return (f"Registros: {len(self.filas):,} | Alcance NOC: {len(self.noc):,}\n"
                f"Cobertura de aperturas: {rango}\n"
                "Los periodos relativos usan la fecha actual de Ciudad de México. El CSV no es una consulta en tiempo real.")

    def medir(self, kpi, periodo, tipo=None, sev=None):
        """Filtra los folios por periodo, tipo y severidad, y aplica el cálculo solicitado."""
        campo = self.campo_periodo if kpi == "TTC" else "FECHA_APERTURA"
        alcance = [r for r in self.noc if (not tipo or normalizar(r["TIPO"]) == normalizar(tipo)) and (not sev or severidad(r) == sev)]
        sin_fecha = sum(fecha(r[campo]) is None for r in alcance)
        seleccion = [r for r in alcance if (f := fecha(r[campo])) and periodo.inicio <= f < periodo.fin]
        return [CALCULOS[kpi](r) for r in seleccion], sin_fecha, campo

    def linea_severidad(self, nombre, mediciones):
        """Prepara el porcentaje y estado de una severidad, sin mostrar tablas extensas."""
        r = resumen_mediciones(mediciones)
        errores = r["sin_datos"] + r["inconsistentes"]
        if not r["evaluables"]:
            if errores:
                return f"{nombre}: Sin datos suficientes para calcular; {errores} folio(s) requieren revisión."
            return f"{nombre}: {SIN_FOLIOS}"
        estado = "Cumple" if r["cumple"] else "No cumple"
        parcial = f" (parcial: {errores} no evaluables)" if errores else ""
        return f"{nombre}: {r['porcentaje']:.2f}% - {estado}{parcial}"

    def consultar(self, consulta, texto):
        """Responde con periodo y resultado por severidad para TTT, TTC o TTN."""
        periodo = periodo_consulta(texto, self.reloj())
        salida = [describir_periodo(periodo)]
        if consulta.get("tipo"):
            salida.append("Tipo: " + consulta["tipo"])
        registros_auditoria = []
        hay_folios = False
        for kpi in consulta["kpis"]:
            ms, sin_fecha, campo = self.medir(kpi, periodo, consulta.get("tipo"), consulta.get("severidad"))
            registros_auditoria.extend((kpi, m) for m in ms)
            hay_folios |= bool(ms)
            salida.append(kpi)
            for sev in ([consulta["severidad"]] if consulta.get("severidad") else ["SF1", "SF2", "SF3", "SF4"]):
                salida.append(self.linea_severidad(sev, [m for m in ms if severidad(m.fila) == sev]))
            desconocidos = sum(severidad(m.fila) == "Sin severidad" for m in ms)
            if desconocidos:
                salida.append(f"Nota: {desconocidos} folio(s) sin severidad reconocida, fuera del desglose SF1–SF4.")
            if sin_fecha:
                salida.append(f"Nota: {sin_fecha} folio(s) del alcance sin fecha de periodo válida; requieren revisión.")
            if campo != "FECHA_APERTURA":
                salida.append("Periodo por fecha de cierre.")
            if consulta["accion"] == "detalle":
                salida.append("Folio | Severidad | Resultado | Minutos")
                for m in ms[:50]:
                    minutos = "N/A" if m.minutos is None else f"{m.minutos:.2f}"
                    salida.append(f"{m.fila['CODIGO_FOLIO']} | {severidad(m.fila)} | {m.estado} | {minutos}")
                if len(ms) > 50:
                    salida.append("Se muestran 50 filas. Usa exportar para obtener todo el detalle.")
            elif consulta["accion"] == "grupos":
                ranking = grupo_trabajo(ms)
                maximo = ranking[0][1]["fuera"] if ranking else 0
                if maximo:
                    nombres = [g for g, r in ranking if r["fuera"] == maximo]
                    salida.append(f"Mayor cantidad fuera de tiempo: {', '.join(nombres)} ({maximo} folios por grupo).")
                else:
                    salida.append("No hay folios evaluables fuera de tiempo para ordenar grupos.")
        if not hay_folios:
            salida.append("No hay información para ese periodo y filtros. Verifica las fechas y reformula la consulta o carga una base actualizada.")
        self.ultima = {"mediciones": registros_auditoria, "consulta": consulta, "periodo": periodo}
        return "\n".join(salida)

    def exportar(self):
        """Guarda el detalle de la última consulta en un CSV para revisión."""
        if not self.ultima:
            return "Primero realiza una consulta de KPI."
        carpeta = self.ruta.parent / "resultados"
        carpeta.mkdir(exist_ok=True)
        ruta = carpeta / ("detalle_kpis_" + self.reloj().strftime("%Y%m%d_%H%M%S_%f") + ".csv")
        campos = ["KPI", "CODIGO_FOLIO", "DESC_PRIORIDAD", "TIPO", "DESC_GRUPO_TRABAJO", "RESULTADO", "MINUTOS", "MOTIVO", "DISCREPANCIA_TTC", "FECHA_APERTURA", "FECHA_INICIO_FALLA", "FECHA_INICIO_TTC", "FECHA_HORA_CIERRE", "TTC", "TTN"]
        with ruta.open("w", encoding="utf-8-sig", newline="") as f:
            w = csv.DictWriter(f, fieldnames=campos)
            w.writeheader()
            for kpi, m in self.ultima["mediciones"]:
                r = {k: m.fila.get(k, "") for k in campos}
                r.update(KPI=kpi, RESULTADO=m.estado, MINUTOS=m.minutos, MOTIVO=m.motivo, DISCREPANCIA_TTC=m.discrepancia)
                # Evita que un texto se interprete como fórmula al abrirlo en Excel.
                w.writerow({k: "'" + v if isinstance(v, str) and v.startswith(("=", "+", "-", "@")) else v for k, v in r.items()})
        return f"Detalle exportado: {ruta}"

    def responder(self, texto):
        """Atiende una pregunta o solicita el dato que falta para poder responder."""
        try:
            c = identificar_intencion(texto)
            if self.pendiente:
                anterior = self.pendiente
                if anterior["accion"] != "estatus" and c.get("kpis"):
                    texto = anterior["texto"] + " " + texto
                    c = identificar_intencion(texto)
                self.pendiente = None
            accion = c["accion"]
            if accion == "despedida":
                self.terminado = True
                respuesta = despedida_bot()
            elif accion == "ayuda":
                respuesta = Intrucciones_bot()
            elif accion == "saludo":
                respuesta = saludo_inicial()
            elif accion == "datos":
                respuesta = self.datos()
            elif accion == "estatus":
                if c.get("folio"):
                    respuesta = estatus_folio(self.filas, c["folio"])
                else:
                    self.pendiente = c
                    respuesta = "¿Qué folio deseas consultar? Ejemplo: 261013-001."
            elif accion == "exportar":
                respuesta = self.exportar()
            elif accion == "cargar":
                respuesta = "Usa la opción de carga del programa para seleccionar el nuevo CSV."
            elif accion in {"kpi", "detalle", "grupos"}:
                if not c["kpis"]:
                    self.pendiente = {**c, "texto": texto}
                    respuesta = "¿Qué indicador deseas consultar: TTT, TTC o TTN?"
                else:
                    respuesta = self.consultar(c, texto)
            else:
                respuesta = "No pude identificar la consulta. Reformúlala; por ejemplo: KPI TTT del 15 de octubre de 2026."
        except (ValueError, OSError) as e:
            respuesta = str(e)
        self.historial.append({"usuario": texto, "bot": respuesta})
        return respuesta


def ejecutar(ruta=None, colab=False):
    """Inicia el bot y mantiene la conversación hasta que el usuario se despide."""
    if wordpunct_tokenize is None:
        print("Instala NLTK: python -m pip install -r requirements.txt")
        return None
    ruta = resolver_ruta_csv(ruta)
    print(Intrucciones_bot())
    correcto, mensaje = valida_estructura(ruta) if valida_archivo(ruta) else (False, "No se encontró Calculo_KPIs.csv.")
    if not correcto:
        print(mensaje)
        if carga_archivo(ruta, colab) is None:
            print("Carga cancelada.")
            return None
    if cargar_bitacora_zip(colab=colab) is None:
        print("No se cargó la carpeta de imágenes; no se puede iniciar el cálculo TTN por folio.")
        return None
    bot = BotKPIs(ruta)
    print("\n" + saludo_inicial())
    while not bot.terminado:
        try:
            texto = input("\nTú: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n" + despedida_bot())
            break
        if not texto:
            continue
        if identificar_intencion(texto)["accion"] == "cargar":
            if carga_archivo(ruta, colab):
                bot = BotKPIs(ruta)
                print("Base actualizada.")
            continue
        print("\nBot: " + bot.responder(texto))
    return bot


def main():
    """Permite iniciar una conversación o ejecutar una consulta desde la terminal."""
    parser = argparse.ArgumentParser(description="Bot de KPIs NOC basado en reglas")
    parser.add_argument("--archivo", type=Path, help="Ruta del CSV; por defecto, junto al programa")
    parser.add_argument("--pregunta", help="Realiza una consulta y termina")
    args = parser.parse_args()
    try:
        if args.pregunta:
            print(BotKPIs(args.archivo).responder(args.pregunta))
        else:
            ejecutar(args.archivo)
    except (ValueError, OSError, RuntimeError) as e:
        parser.exit(1, str(e) + "\n")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile -a bot_kpis.py
# Extensión para el cálculo individual de TTN por imagen.
# Requiere un dataset visual etiquetado: train/ y val/, cada uno con las clases
# con_notificacion_ctt/ y sin_notificacion_ctt/.
import os

TTN_LIMITES = {"SF1": 10, "SF2": 10, "SF3": 30, "SF4": 40}
TTN_MODELO_PATH = Path("resnet50_ttn.pth")
TTN_CLASES = ["con_notificacion_ctt", "sin_notificacion_ctt"]
BITACORA_DIR = None
TTN_EXT_IMAGENES = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff", ".webp"}
DIRECTORIO_DOCUMENTOS = Path(os.environ.get("BOT_DOCUMENTS_DIR", "/content/documentos"))


def buscar_documento(nombre, directorio=None):
    """Busca recursivamente un archivo dentro del ZIP de documentos ya extraído."""
    raiz = Path(directorio or DIRECTORIO_DOCUMENTOS)
    if not raiz.is_dir():
        raise FileNotFoundError(f"No existe la carpeta de documentos: {raiz.resolve()}")
    coincidencias = [p for p in raiz.rglob("*") if p.is_file() and p.name.lower() == nombre.lower()]
    if not coincidencias:
        raise FileNotFoundError(f"No encontré {nombre} dentro de {raiz.resolve()}.")
    if len(coincidencias) > 1:
        raise ValueError(f"Encontré más de un archivo llamado {nombre} en documentos.zip. Deja una sola copia.")
    return coincidencias[0]


def _extraer_zip_seguro(ruta_zip, destino):
    """Extrae un ZIP y bloquea rutas que intenten salir del directorio destino."""
    import zipfile
    raiz = Path(destino).resolve()
    raiz.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(ruta_zip) as archivo:
        for miembro in archivo.infolist():
            objetivo = (raiz / miembro.filename).resolve()
            if objetivo != raiz and raiz not in objetivo.parents:
                raise ValueError(f"El ZIP contiene una ruta no permitida: {miembro.filename}")
        archivo.extractall(raiz)
    return raiz


def cargar_bitacora_zip(colab=False, entrada=input):
    """Prepara bitacora.zip desde documentos.zip; en local conserva la carga por ruta."""
    global BITACORA_DIR
    import tempfile
    if colab:
        try:
            ruta_zip = buscar_documento("bitacora.zip")
        except (FileNotFoundError, ValueError) as exc:
            print(f"No pude localizar bitacora.zip en documentos.zip: {exc}")
            return None
        BITACORA_DIR = _extraer_zip_seguro(ruta_zip, DIRECTORIO_DOCUMENTOS / "bitacora_extraida")
    else:
        try:
            ruta_zip = buscar_documento("bitacora.zip")
            BITACORA_DIR = _extraer_zip_seguro(ruta_zip, DIRECTORIO_DOCUMENTOS / "bitacora_extraida")
        except FileNotFoundError:
            # Compatibilidad con el uso del módulo sin documentos.zip.
            respuesta = entrada("Ruta de bitacora.zip o carpeta con las imágenes (o cancelar): ").strip().strip("\"'")
            if normalizar(respuesta) in {"", "cancelar", "salir"}:
                return None
            ruta = Path(respuesta).expanduser()
            if ruta.is_dir():
                BITACORA_DIR = ruta.resolve()
            elif ruta.is_file() and ruta.suffix.lower() == ".zip":
                temporal = Path(tempfile.mkdtemp(prefix="bitacora_ttn_"))
                BITACORA_DIR = _extraer_zip_seguro(ruta, temporal / "imagenes")
            else:
                print("Indica un archivo .zip válido o una carpeta de imágenes.")
                return None
    imagenes = [p for p in BITACORA_DIR.rglob("*") if p.is_file() and p.suffix.lower() in TTN_EXT_IMAGENES]
    if not imagenes:
        print("El ZIP/carpeta no contiene imágenes PNG, JPG, JPEG, BMP, TIF o WEBP.")
        BITACORA_DIR = None
        return None
    print(f"Bitácora lista: {len(imagenes)} imagen(es) disponibles en {BITACORA_DIR}.")
    return BITACORA_DIR


def buscar_imagen_folio(folio, carpeta=None):
    """Encuentra una imagen cuyo nombre contiene exactamente el folio consultado."""
    carpeta = Path(carpeta or BITACORA_DIR) if (carpeta or BITACORA_DIR) else None
    if carpeta is None or not carpeta.is_dir():
        return None, "No está cargado bitacora.zip. Inicia nuevamente el bot y cárgalo después del CSV."
    coincidencias = []
    for ruta in carpeta.rglob("*"):
        if ruta.is_file() and ruta.suffix.lower() in TTN_EXT_IMAGENES:
            folios_nombre = re.findall(r"\d{6}-\d{3,}", ruta.stem)
            if folio.upper() in {f.upper() for f in folios_nombre}:
                coincidencias.append(ruta)
    if not coincidencias:
        return None, f"No encontré una imagen cuyo nombre incluya el folio {folio}. Verifica el nombre del archivo dentro de bitacora.zip."
    if len(coincidencias) > 1:
        nombres = ", ".join(p.name for p in coincidencias[:8])
        return None, f"Encontré varias imágenes para {folio}: {nombres}. Deja una sola imagen por folio en bitacora.zip."
    return coincidencias[0], None


def crear_resnet50_ttn(num_clases, pretrained=True):
    """Construye ResNet50 con transferencia de aprendizaje y una cabeza propia."""
    import torch.nn as nn
    from torchvision import models
    pesos = models.ResNet50_Weights.DEFAULT if pretrained else None
    modelo = models.resnet50(weights=pesos)
    for parametro in modelo.parameters():
        parametro.requires_grad = False
    modelo.fc = nn.Linear(modelo.fc.in_features, num_clases)
    for parametro in modelo.fc.parameters():
        parametro.requires_grad = True
    return modelo


def evaluacion_visual(modelo, cargador, clases, dispositivo="cpu"):
    """Calcula Accuracy, Precision, Recall, F1-score y grafica la matriz de confusión."""
    import torch
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                 f1_score, confusion_matrix, classification_report)
    modelo.eval()
    reales, predichas = [], []
    with torch.no_grad():
        for imagenes, etiquetas in cargador:
            logits = modelo(imagenes.to(dispositivo))
            predichas.extend(logits.argmax(dim=1).cpu().tolist())
            reales.extend(etiquetas.tolist())
    indices = list(range(len(clases)))
    matriz = confusion_matrix(reales, predichas, labels=indices)
    resultados = {
        "Accuracy": accuracy_score(reales, predichas),
        "Precision_macro": precision_score(reales, predichas, average="macro", zero_division=0),
        "Recall_macro": recall_score(reales, predichas, average="macro", zero_division=0),
        "F1_score_macro": f1_score(reales, predichas, average="macro", zero_division=0),
        "Matriz_de_confusion": matriz.tolist(),
        "Reporte_por_clase": classification_report(reales, predichas, target_names=clases, zero_division=0),
    }
    print("Accuracy:", f"{resultados['Accuracy']:.4f}")
    print("Precision (macro):", f"{resultados['Precision_macro']:.4f}")
    print("Recall (macro):", f"{resultados['Recall_macro']:.4f}")
    print("F1-score (macro):", f"{resultados['F1_score_macro']:.4f}")
    print("\nReporte por clase:\n", resultados["Reporte_por_clase"])
    fig, ax = plt.subplots(figsize=(7, 6))
    sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False,
                xticklabels=clases, yticklabels=clases, ax=ax)
    ax.set_xlabel("Predicción del modelo")
    ax.set_ylabel("Etiqueta real")
    ax.set_title("Matriz de confusión - ResNet50 TTN")
    fig.tight_layout()
    plt.show()
    return resultados


def entrenar_resnet50_ttn(carpeta_dataset="dataset_ttn", epocas=5, batch_size=16):
    """Entrena y evalúa ResNet50 con carpetas ImageFolder train/ y val/."""
    import torch
    from torch import nn
    from torch.utils.data import DataLoader
    from torchvision import transforms
    from torchvision.datasets import ImageFolder
    from torchvision.models import ResNet50_Weights
    raiz = Path(carpeta_dataset)
    if not raiz.is_dir():
        raise FileNotFoundError(f"No existe la carpeta extraída del dataset: {raiz.resolve()}")
    # Admite ZIPs con una carpeta superior adicional, por ejemplo:
    # dataset_ttn/dataset_ttn/train/... y dataset_ttn/dataset_ttn/val/...
    candidatas = [raiz] + [p for p in raiz.rglob("*") if p.is_dir()]
    raices_validas = [p for p in candidatas if (p / "train").is_dir() and (p / "val").is_dir()]
    if not raices_validas:
        muestra = "\n".join(str(p.relative_to(raiz)) for p in list(raiz.rglob("*"))[:30])
        raise FileNotFoundError(
            "No encontré las carpetas train y val dentro del ZIP extraído. "
            "Se espera train/<clase>/*.jpg y val/<clase>/*.jpg. "
            f"Contenido encontrado bajo {raiz.resolve()}:\n{muestra or '(carpeta vacía)'}"
        )
    raiz = min(raices_validas, key=lambda p: len(p.parts))
    print(f"Dataset detectado en: {raiz.resolve()}")
    transform = ResNet50_Weights.DEFAULT.transforms()
    train_ds = ImageFolder(raiz / "train", transform=transform)
    val_ds = ImageFolder(raiz / "val", transform=transform)
    if train_ds.classes != val_ds.classes:
        raise ValueError("Las clases de train y val deben coincidir.")
    if train_ds.classes != TTN_CLASES:
        raise ValueError("Las carpetas deben llamarse exactamente: " + ", ".join(TTN_CLASES))
    from collections import Counter
    n_train, n_val = Counter(train_ds.targets), Counter(val_ds.targets)
    if any(n_train[i] < 2 or n_val[i] < 1 for i in range(len(train_ds.classes))):
        raise ValueError("Usa al menos 2 imágenes por clase en train y 1 por clase en val; para resultados confiables se necesitan muchas más.")
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    modelo = crear_resnet50_ttn(len(train_ds.classes), pretrained=True).to(dispositivo)
    optimizador = torch.optim.AdamW((p for p in modelo.parameters() if p.requires_grad), lr=1e-3)
    perdida = nn.CrossEntropyLoss()
    train_dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    val_dl = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
    for epoca in range(epocas):
        modelo.train()
        acumulada, aciertos, total = 0.0, 0, 0
        for imagenes, etiquetas in train_dl:
            imagenes, etiquetas = imagenes.to(dispositivo), etiquetas.to(dispositivo)
            optimizador.zero_grad()
            logits = modelo(imagenes)
            loss = perdida(logits, etiquetas)
            loss.backward()
            optimizador.step()
            acumulada += loss.item() * etiquetas.size(0)
            aciertos += (logits.argmax(1) == etiquetas).sum().item()
            total += etiquetas.size(0)
        print(f"Época {epoca + 1}/{epocas} | pérdida={acumulada / total:.4f} | accuracy_train={aciertos / total:.4f}")
    metricas = evaluacion_visual(modelo, val_dl, train_ds.classes, dispositivo)
    torch.save({"state_dict": modelo.cpu().state_dict(), "clases": train_ds.classes}, TTN_MODELO_PATH)
    print(f"Modelo guardado en: {TTN_MODELO_PATH.resolve()}")
    return metricas


def _cargar_modelo_ttn(ruta_modelo=TTN_MODELO_PATH):
    import torch
    from torchvision.models import ResNet50_Weights
    if not Path(ruta_modelo).is_file():
        raise FileNotFoundError("No está entrenado ResNet50 TTN. Ejecuta primero entrenar_resnet50_ttn() con un dataset etiquetado.")
    dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
    checkpoint = torch.load(ruta_modelo, map_location=dispositivo)
    clases = checkpoint["clases"]
    modelo = crear_resnet50_ttn(len(clases), pretrained=False)
    modelo.load_state_dict(checkpoint["state_dict"])
    modelo.to(dispositivo).eval()
    transform = ResNet50_Weights.DEFAULT.transforms()
    return modelo, transform, clases, dispositivo


def _fecha_hora_ttn(texto):
    """Lee fechas DD/mmm/YYYY hh:mm AM/PM, DD/MM/YYYY y marcas de tiempo ISO."""
    meses = {"ene": 1, "jan": 1, "feb": 2, "mar": 3, "abr": 4, "apr": 4,
             "may": 5, "jun": 6, "jul": 7, "ago": 8, "aug": 8, "sep": 9,
             "oct": 10, "nov": 11, "dic": 12, "dec": 12}
    patron = re.compile(r"\b(\d{1,2})[/-]([A-Za-zÁÉÍÓÚáéíóú]{3,}|\d{1,2})[/-](\d{4})\s+(\d{1,2}):(\d{2})(?::(\d{2}))?\s*(AM|PM)?", re.I)
    patron_iso = re.compile(r"\b(\d{4})-(\d{2})-(\d{2})[ T](\d{1,2}):(\d{2})(?::(\d{2}))?")
    encontrados = []
    for m in patron.finditer(texto):
        dia, mes_txt, anio, hora, minuto, segundo, meridiano = m.groups()
        mes = int(mes_txt) if mes_txt.isdigit() else meses.get(normalizar(mes_txt)[:3])
        if not mes:
            continue
        hora = int(hora)
        if meridiano:
            hora = hora % 12 + (12 if meridiano.upper() == "PM" else 0)
        try:
            encontrados.append(datetime(int(anio), mes, int(dia), hora, int(minuto), int(segundo or 0)))
        except ValueError:
            pass
    for m in patron_iso.finditer(texto):
        anio, mes, dia, hora, minuto, segundo = m.groups()
        try:
            encontrados.append(datetime(int(anio), int(mes), int(dia), int(hora), int(minuto), int(segundo or 0)))
        except ValueError:
            pass
    return encontrados


def _analizar_imagen_ttn(imagen, ruta_csv, folio_solicitado):
    """Clasifica la captura con CNN y extrae por OCR folio, tiempos y severidad."""
    import torch
    import pytesseract
    from PIL import Image, ImageOps
    modelo, transform, clases, dispositivo = _cargar_modelo_ttn()
    imagen = ImageOps.exif_transpose(imagen).convert("RGB")
    entrada = transform(imagen).unsqueeze(0).to(dispositivo)
    with torch.no_grad():
        prob = torch.softmax(modelo(entrada), dim=1)[0]
    indice = int(prob.argmax().item())
    clase_cnn, confianza = clases[indice], float(prob[indice].item())
    grande = ImageOps.grayscale(imagen)
    if grande.width < 1800:
        factor = min(3, 1800 / grande.width)
        grande = grande.resize((int(grande.width * factor), int(grande.height * factor)))
    texto = pytesseract.image_to_string(grande, lang="spa+eng", config="--psm 11")
    if "NOTIFICACION_CCT" not in texto.upper():
        return (f"No encontré la cadena NOTIFICACION_CCT en la imagen.\n"
                f"Clasificación CNN: {clase_cnn} ({confianza:.1%}). Revisa que la captura incluya el comentario de notificación.")
    folios_imagen = {f.upper() for f in re.findall(r"\b\d{6}-\d{3,}\b", texto)}
    if not folios_imagen:
        return f"Detecté NOTIFICACION_CCT, pero no pude leer ningún folio en la imagen.\nClasificación CNN: {clase_cnn} ({confianza:.1%})."
    solicitado = folio_solicitado.strip().upper()
    if solicitado not in folios_imagen:
        detectados = ", ".join(sorted(folios_imagen))
        return (f"El folio solicitado ({solicitado}) no coincide con los folios detectados en la imagen "
                f"({detectados}). No calculé el TTN. Verifica que subiste la captura correcta.")
    folio = solicitado
    filas = leer_csv(ruta_csv)
    registro = next((r for r in filas if r["CODIGO_FOLIO"].upper() == folio.upper()), None)
    if not registro:
        return f"Detecté NOTIFICACION_CCT y el folio {folio}, pero no aparece en Calculo_KPIs.csv."
    msev = re.search(r"\bSF\s*([1-4])\b", registro.get("DESC_PRIORIDAD", ""), re.I)
    if not msev:
        return f"No pude identificar SF1-SF4 en DESC_PRIORIDAD para el folio {folio}."
    severidad_ttn = "SF" + msev.group(1)
    lineas = texto.splitlines()
    hora_notificacion = None
    for i, linea in enumerate(lineas):
        if "NOTIFICACION_CCT" in linea.upper():
            for candidata in reversed(lineas[max(0, i - 4):i + 1]):
                fechas = _fecha_hora_ttn(candidata)
                if fechas:
                    hora_notificacion = fechas[0]
                    break
            if hora_notificacion:
                break
    todas = _fecha_hora_ttn(texto)
    candidatas_inicio = [f for f in todas if f != hora_notificacion] if hora_notificacion else todas
    hora_inicio = min(candidatas_inicio) if candidatas_inicio else None
    if not hora_notificacion or not hora_inicio:
        return (f"Folio: {folio} | Severidad: {severidad_ttn}\n"
                "No pude leer la hora de generación de alarma y la hora del encabezado NOTIFICACION_CCT. Revisa la imagen.")
    minutos = (hora_notificacion - hora_inicio).total_seconds() / 60
    if minutos < 0:
        return f"Folio: {folio}. Las fechas OCR quedaron en orden inverso; revisa la imagen antes de concluir el SLA."
    limite = TTN_LIMITES[severidad_ttn]
    resultado = "Dentro de SLA" if minutos <= limite else "Fuera de SLA"
    return (f"Folio: {folio}\nSeveridad: {severidad_ttn} (Calculo_KPIs.csv / DESC_PRIORIDAD)\n"
            f"Hora de generación: {hora_inicio:%d/%m/%Y %H:%M:%S}\n"
            f"Hora de notificación: {hora_notificacion:%d/%m/%Y %H:%M:%S}\n"
            f"Cadena NOTIFICACION_CCT: identificada\nTiempo: {minutos:.2f} min (límite SF: {limite} min)\n"
            f"{resultado}\nClasificación CNN: {clase_cnn} ({confianza:.1%})")


def calculoTTN(ruta_csv=None, folio_solicitado=None):
    """Busca en bitacora.zip la imagen del folio y calcula su TTN.

    El ZIP se solicita una sola vez al iniciar el bot, después del CSV.
    La clasificación requiere el modelo ResNet50 entrenado previamente.
    """
    from PIL import Image
    if not folio_solicitado:
        folio_solicitado = input("Indica el número de FG/TT que deseas validar (formato 261104-001): ").strip()
    folio_solicitado = folio_solicitado.strip().upper()
    if not re.fullmatch(r"\d{6}-\d{3,}", folio_solicitado):
        return "El número de FG/TT debe tener el formato 261104-001. No calculé el TTN."
    ruta_img, error = buscar_imagen_folio(folio_solicitado)
    if error:
        return error
    imagen = Image.open(ruta_img).convert("RGB")
    ruta_csv = resolver_ruta_csv(ruta_csv)
    try:
        respuesta = _analizar_imagen_ttn(imagen, ruta_csv, folio_solicitado)
    except FileNotFoundError as e:
        respuesta = str(e)
    except Exception as e:
        respuesta = f"No pude completar el cálculo TTN: {e}"
    print(respuesta)
    try:
        import google.colab  # noqa: F401
        es_colab = True
    except ImportError:
        es_colab = False
    if es_colab:
        from IPython.display import display
        display(imagen)
    return respuesta


_identificar_intencion_original_ttn = identificar_intencion
def identificar_intencion(texto):
    """Distingue el cálculo TTN puntual con imagen de una consulta agregada TTN."""
    normal = normalizar(texto)
    tiene_ttn = bool(re.search(r"\bttn\b", normal)) or "time to notify" in normal
    solicita_calculo = bool(re.search(r"\b(calculo|calcular|calcula|evaluar|evalua)\b", normal))
    puntual = bool(re.search(r"\b(fg|folio|imagen|captura|alarma)\b", normal) or
                   re.search(r"\b\d{6}-\d{3,}\b", normal))
    folio = re.search(r"\b\d{6}-\d{3,}\b", normal)
    # La mención de TTN junto con un folio identifica un cálculo individual,
    # aunque el usuario no escriba explícitamente "calcular".
    if tiene_ttn and ((solicita_calculo and puntual) or folio):
        return {"accion": "calculo_ttn_imagen", "folio_solicitado": folio.group(0) if folio else None}
    return _identificar_intencion_original_ttn(texto)


_responder_original_ttn = BotKPIs.responder
def _responder_con_calculo_ttn(self, texto):
    intencion = identificar_intencion(texto)
    if intencion.get("accion") == "calculo_ttn_imagen":
        respuesta = calculoTTN(self.ruta, folio_solicitado=intencion.get("folio_solicitado"))
        self.historial.append({"usuario": texto, "bot": respuesta})
        return respuesta
    return _responder_original_ttn(self, texto)


BotKPIs.responder = _responder_con_calculo_ttn


# Clasificador de intenciones conversacionales: TF-IDF + SVM lineal.
MODELO_INTENCIONES = None
STOPWORDS_ES = set("a al algo algunas algunos ante antes como con contra cual cuando de del desde donde durante e el ella ellas ellos en entre era esa esas ese eso esos esta estaba estaban estado estas este esto estos fue fueron ha hacia hasta hay la las le les lo los mas me mi mis mucha muchas mucho muchos muy ni no nos o os otra otras otro otros para pero poco por porque que quien se sin sobre su sus te tiene tu tus un una unas uno unos y ya".split())


def cargar_frases_intenciones(ruta="frases_intenciones.csv"):
    """Carga un CSV con exactamente dos columnas: frase e intencion."""
    import csv
    import io
    ruta = Path(ruta)
    if not ruta.is_file():
        raise FileNotFoundError(f"No encontré el archivo de frases: {ruta.resolve()}")
    contenido = ruta.read_bytes()
    texto_csv = None
    for codificacion in ("utf-8-sig", "cp1252", "latin-1"):
        try:
            texto_csv = contenido.decode(codificacion)
            break
        except UnicodeDecodeError:
            continue
    if texto_csv is None:
        raise ValueError("No pude decodificar frases_intenciones.csv. Guárdalo como UTF-8 o CSV de Windows.")
    lector = csv.DictReader(io.StringIO(texto_csv, newline=""))
    encabezados = [c.strip().lower() for c in (lector.fieldnames or [])]
    if encabezados != ["frase", "intencion"]:
        raise ValueError("El CSV debe tener exactamente las columnas frase,intencion, en ese orden.")
    filas = list(lector)
    frases, intenciones = [], []
    for numero, fila in enumerate(filas, start=2):
        frase = (fila.get("frase") or "").strip()
        intencion = (fila.get("intencion") or "").strip().lower()
        if not frase or not intencion:
            raise ValueError(f"La fila {numero} tiene frase o intencion vacía.")
        frases.append(frase)
        intenciones.append(intencion)
    if len(set(intenciones)) < 2:
        raise ValueError("Se requieren al menos dos clases de intención.")
    return frases, intenciones


def normalizacion(texto):
    """Pasa a minúsculas, quita acentos y normaliza espacios."""
    texto = unicodedata.normalize("NFKD", str(texto).lower())
    texto = "".join(c for c in texto if not unicodedata.combining(c))
    return " ".join(texto.split())


def tokenizacion(texto):
    """Separa texto normalizado en palabras y números."""
    return re.findall(r"[a-z]+|\d+", normalizacion(texto))


def eliminacion_stopwords(tokens):
    """Elimina palabras funcionales comunes del español."""
    return [token for token in tokens if token not in STOPWORDS_ES and len(token) > 1]


def stemming(tokens):
    """Reduce las palabras a su raíz con Snowball para español."""
    from nltk.stem.snowball import SnowballStemmer
    stemmer = SnowballStemmer("spanish")
    return [stemmer.stem(token) for token in tokens]


def preprocesar_intencion(texto):
    """Aplica normalización, tokenización, stopwords y stemming."""
    return stemming(eliminacion_stopwords(tokenizacion(texto)))


def crear_vectorizador_tfidf():
    """Crea TF-IDF para unigramas y bigramas usando el preprocesamiento anterior."""
    from sklearn.feature_extraction.text import TfidfVectorizer
    return TfidfVectorizer(tokenizer=preprocesar_intencion, token_pattern=None,
                           lowercase=False, ngram_range=(1, 2), sublinear_tf=True)


def representar_tfidf(frases):
    """Ajusta TF-IDF y devuelve la matriz dispersa y el vectorizador."""
    vectorizador = crear_vectorizador_tfidf()
    matriz = vectorizador.fit_transform(frases)
    return matriz, vectorizador


def crear_pipeline_svm():
    """Crea la secuencia TF-IDF más SVM lineal."""
    from sklearn.pipeline import Pipeline
    from sklearn.svm import LinearSVC
    return Pipeline([
        ("tfidf", crear_vectorizador_tfidf()),
        ("svm", LinearSVC(C=1.0, class_weight="balanced", random_state=42)),
    ])


def entrenar_clasificador_svm(frases, intenciones):
    """Entrena un modelo SVM de intenciones con todas las frases indicadas."""
    modelo = crear_pipeline_svm()
    return modelo.fit(frases, intenciones)


def evaluar_svm_intenciones(modelo, frases_prueba, intenciones_reales, mostrar=True):
    """Calcula Accuracy, Precision, Recall, F1-score y matriz de confusión."""
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                                 f1_score, confusion_matrix, classification_report)
    predicciones = modelo.predict(frases_prueba)
    etiquetas = sorted(set(intenciones_reales) | set(predicciones))
    matriz = confusion_matrix(intenciones_reales, predicciones, labels=etiquetas)
    metricas = {
        "accuracy": accuracy_score(intenciones_reales, predicciones),
        "precision_macro": precision_score(intenciones_reales, predicciones, average="macro", zero_division=0),
        "recall_macro": recall_score(intenciones_reales, predicciones, average="macro", zero_division=0),
        "f1_macro": f1_score(intenciones_reales, predicciones, average="macro", zero_division=0),
        "reporte": classification_report(intenciones_reales, predicciones, labels=etiquetas, zero_division=0),
        "etiquetas": etiquetas,
        "matriz_confusion": matriz,
        "predicciones": predicciones,
    }
    if mostrar:
        print(f"Accuracy: {metricas['accuracy']:.4f}")
        print(f"Precision macro: {metricas['precision_macro']:.4f}")
        print(f"Recall macro: {metricas['recall_macro']:.4f}")
        print(f"F1-score macro: {metricas['f1_macro']:.4f}")
        print("\nReporte por intención:\n", metricas["reporte"])
        fig, ax = plt.subplots(figsize=(10, 8))
        sns.heatmap(matriz, annot=True, fmt="d", cmap="Blues", cbar=False,
                    xticklabels=etiquetas, yticklabels=etiquetas, ax=ax)
        ax.set_xlabel("Intención predicha")
        ax.set_ylabel("Intención real")
        ax.set_title("Matriz de confusión - SVM de intenciones")
        plt.xticks(rotation=45, ha="right")
        fig.tight_layout()
        plt.show()
    return metricas


def entrenar_intenciones(ruta_csv="frases_intenciones.csv", test_size=0.25):
    """Carga frases, evalúa una partición estratificada y registra el SVM final."""
    global MODELO_INTENCIONES
    from sklearn.model_selection import train_test_split
    frases, intenciones = cargar_frases_intenciones(ruta_csv)
    try:
        X_train, X_test, y_train, y_test = train_test_split(
            frases, intenciones, test_size=test_size, random_state=42, stratify=intenciones
        )
    except ValueError as exc:
        raise ValueError("No hay suficientes frases por intención para crear una partición estratificada. Añade más ejemplos por clase.") from exc
    modelo_evaluacion = entrenar_clasificador_svm(X_train, y_train)
    metricas = evaluar_svm_intenciones(modelo_evaluacion, X_test, y_test)
    # Para usarlo en el chatbot, se vuelve a entrenar con todas las frases disponibles.
    MODELO_INTENCIONES = entrenar_clasificador_svm(frases, intenciones)
    print(f"\nFrases: {len(frases)} | Entrenamiento de evaluación: {len(X_train)} | Prueba: {len(X_test)}")
    return {"modelo": MODELO_INTENCIONES, "metricas": metricas}


_identificar_intencion_con_ttn = identificar_intencion
def identificar_intencion(texto):
    """Usa SVM para la intención general y conserva extracción de campos del bot."""
    resultado = _identificar_intencion_con_ttn(texto)
    if resultado.get("accion") == "calculo_ttn_imagen" or MODELO_INTENCIONES is None:
        return resultado
    predicha = str(MODELO_INTENCIONES.predict([texto])[0]).strip().lower()
    # Si el SVM responde desconocida, se conserva la regla existente como respaldo.
    if predicha != "desconocida":
        # Una consulta explícita de estatus con folio debe conservar esa ruta;
        # el modelo puede confundir el número con una consulta agregada de KPI.
        tiene_folio = bool(re.search(r"\b\d{6}-\d{3,}\b", normalizar(texto)))
        if not (resultado.get("accion") == "estatus" and tiene_folio and predicha != "estatus"):
            resultado["accion"] = predicha
    # Algunas reglas de reconocimiento retornan temprano (por ejemplo, al ver
    # un folio) y no incluyen los campos de una consulta KPI. Mantener un esquema
    # completo evita KeyError cuando el SVM clasifica la frase como KPI/detalle.
    resultado.setdefault("kpis", [])
    resultado.setdefault("tipo", None)
    resultado.setdefault("severidad", None)
    if resultado["accion"] in {"kpi", "detalle", "grupos"} and not resultado["kpis"]:
        normal = normalizar(texto)
        resultado["kpis"] = [nombre for nombre in CALCULOS if re.search(rf"\b{nombre.lower()}\b", normal)]
    return resultado


In [ ]:
from bot_kpis import buscar_documento, _extraer_zip_seguro, entrenar_resnet50_ttn

ruta_dataset_zip = buscar_documento('dataset_ttn.zip')
carpeta_dataset = _extraer_zip_seguro(ruta_dataset_zip, Path('/content/documentos/dataset_ttn_extraido'))
metricas_visual = entrenar_resnet50_ttn(carpeta_dataset, epocas=5)


In [ ]:
from bot_kpis import entrenar_intenciones, buscar_documento

ruta_frases = buscar_documento('frases_intenciones.csv')
resultado_intenciones = entrenar_intenciones(ruta_frases)
modelo_intenciones_svm = resultado_intenciones['modelo']


In [ ]:
from pathlib import Path
from bot_kpis import ejecutar, buscar_documento

# Ejecuta con la base ya incluida en documentos.zip.
ruta_kpis = buscar_documento("Calculo_KPIs.csv")
bot = ejecutar(ruta_kpis, colab=True)
